# 12 — Holdout evaluation leaderboard (saved table)

**Summary:** For each `MODEL_ID`, loads **postprocessed** holdout predictions produced by notebooks **08** and **09** (first matching eval folder: tuning, best_extended, curriculum), scores **all** holdout rows, joins `eval_loss` from `round1_results.csv`–`round4_results.csv` when `registry_model_id` is present, flattens selected fields from the registry training config, and writes a CSV under `<WORKFLOW_ROOT>/evaluations/holdout_leaderboard/`.

**Prerequisites:** Run **03** (splits) and **08–09** with the same `RUN_PROFILE_ID` so each `model_id` has `predictions_post_<POSTPROCESS_METHOD>.csv` under the profile eval roots.

#### Colab / install

In [4]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!pip -q install pandas cairosvg pillow lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.3 MB/s eta 0:00:00


#### Parameters + run

In [8]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path('/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RUN_PROFILE_ID = 'debug' #'default'
WORKFLOW_ROOT = PROJECT_DIR / 'outputs' / 'workflow_runs' / RUN_PROFILE_ID
MODELS_ROOT = WORKFLOW_ROOT / 'models'
EXPERIMENT_ROOT = WORKFLOW_ROOT / 'lora_tuning_workflow'
LEADERBOARD_DIR = WORKFLOW_ROOT / 'evaluations' / 'holdout_leaderboard'
LEADERBOARD_CSV = LEADERBOARD_DIR / 'holdout_metrics_by_model.csv'

# Empty list = all model_id rows in model_registry.csv for this RUN_PROFILE_ID.
# Or set explicit ids, e.g. MODEL_IDS = ['abc123deadbeef'].
MODEL_IDS = []

POSTPROCESS_METHOD = 'current_default_sanitizer'

from src.eval.holdout_leaderboard import build_holdout_leaderboard_df
from src.training.lora.registry import load_registry

_reg = load_registry(PROJECT_DIR, models_root=MODELS_ROOT)
if MODEL_IDS:
    model_ids_to_run = [str(x) for x in MODEL_IDS]
else:
    if _reg.empty:
        raise ValueError(
            f'No models in registry at {MODELS_ROOT / "model_registry.csv"}. '
            'Train/register models first, or set MODEL_IDS explicitly.'
        )
    model_ids_to_run = _reg['model_id'].astype(str).tolist()

print(f'Leaderboard: {len(model_ids_to_run)} model_id(s)')

df = build_holdout_leaderboard_df(
    PROJECT_DIR,
    WORKFLOW_ROOT,
    model_ids_to_run,
    POSTPROCESS_METHOD,
    workflow_root=WORKFLOW_ROOT,
    experiment_root=EXPERIMENT_ROOT,
)
LEADERBOARD_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(LEADERBOARD_CSV, index=False)
print('Wrote', LEADERBOARD_CSV, df.shape)
display(df)

Leaderboard: 19 model_id(s)
Wrote /content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL/outputs/workflow_runs/debug/evaluations/holdout_leaderboard/holdout_metrics_by_model.csv (19, 27)


,model_id,tuning_stage,curriculum,adapter_dir,cfg_base_model_id,cfg_max_seq_length,cfg_learning_rate,cfg_max_steps,cfg_lora_r,cfg_lora_alpha,...,manifest_base_model_id,n_holdout_rows,avg_target_char_len,svg_open_rate,svg_close_rate,xml_parse_rate,render_rate,avg_pred_char_len,submission_valid_rate,avg_pred_path_count
0,c69ca52c5f4b48e2,round1,False,outputs/workflow_runs/debug/models/lora_model_...,Qwen/Qwen2.5-Coder-1.5B-Instruct,256,0.0001,10,16,32,...,Qwen/Qwen2.5-Coder-1.5B-Instruct,5,2144.0,1.0,1.0,1.0,1.0,93.0,1.0,0.0
1,8126949c043642d1,round1,False,outputs/workflow_runs/debug/models/lora_model_...,Qwen/Qwen2.5-Coder-1.5B-Instruct,256,0.0001,15,16,32,...,Qwen/Qwen2.5-Coder-1.5B-Instruct,5,2144.0,1.0,1.0,1.0,1.0,93.0,1.0,0.0
2,38e6138a9d2e4d4d,round1,False,outputs/workflow_runs/debug/models/lora_model_...,Qwen/Qwen2.5-Coder-1.5B-Instruct,256,0.0002,10,16,32,...,Qwen/Qwen2.5-Coder-1.5B-Instruct,5,2144.0,1.0,1.0,1.0,1.0,93.0,1.0,0.0
3,693e216be194476b,round1,False,outputs/workflow_runs/debug/models/lora_model_...,Qwen/Qwen2.5-Coder-1.5B-Instruct,256,0.0002,15,16,32,...,Qwen/Qwen2.5-Coder-1.5B-Instruct,5,2144.0,1.0,1.0,1.0,1.0,93.0,1.0,0.0
4,0e1ce9f2a98d4d5f,round1,False,outputs/workflow_runs/debug/models/lora_model_...,Qwen/Qwen2.5-Coder-1.5B-Instruct,257,0.0001,10,16,32,...,Qwen/Qwen2.5-Coder-1.5B-Instruct,5,2144.0,1.0,1.0,1.0,1.0,93.0,1.0,0.0
5,1bb2869b057e49f9,round1,False,outputs/workflow_runs/debug/models/lora_model_...,Qwen/Qwen2.5-Coder-1.5B-Instruct,257,0.0001,15,16,32,...,Qwen/Qwen2.5-Coder-1.5B-Instruct,5,2144.0,1.0,1.0,1.0,1.0,93.0,1.0,0.0
6,f65d06c7e5a541e9,round1,False,outputs/workflow_runs/debug/models/lora_model_...,Qwen/Qwen2.5-Coder-1.5B-Instruct,257,0.0002,10,16,32,...,Qwen/Qwen2.5-Coder-1.5B-Instruct,5,2144.0,1.0,1.0,1.0,1.0,93.0,1.0,0.0
7,53b4c946779848eb,round1,False,outputs/workflow_runs/debug/models/lora_model_...,Qwen/Qwen2.5-Coder-1.5B-Instruct,257,0.0002,15,16,32,...,Qwen/Qwen2.5-Coder-1.5B-Instruct,5,2144.0,1.0,1.0,1.0,1.0,93.0,1.0,0.0
8,2a215b96347e4323,round2,False,outputs/workflow_runs/debug/models/lora_model_...,Qwen/Qwen2.5-Coder-1.5B-Instruct,256,0.0002,15,8,16,...,Qwen/Qwen2.5-Coder-1.5B-Instruct,5,2144.0,1.0,1.0,1.0,1.0,93.0,1.0,0.0
9,2f14777da9e24121,round2,False,outputs/workflow_runs/debug/models/lora_model_...,Qwen/Qwen2.5-Coder-1.5B-Instruct,256,0.0002,15,16,32,...,Qwen/Qwen2.5-Coder-1.5B-Instruct,5,2144.0,1.0,1.0,1.0,1.0,93.0,1.0,0.0
